# 实验课 04：单层索引器第一次学会选择了吗？

本课读 G3a 的真实保存结果：Qwen3-0.6B 主体冻结，只训练第 14 层的一个分组索引器。新 T1 资料 48 份训练、16 份留出；固定 P0、r=32、ρ=1/8。

默认执行只读归档、画图和做一个 CPU 张量检查，不下载模型、不训练、不启动 GPU。它回答“更像完整注意力了吗”，不回答“整模型答题恢复了吗”。

In [ ]:
from pathlib import Path
import importlib.util, hashlib, json, ast, sys
from IPython.display import display, Markdown, Code
ROOT = Path.cwd()
if not (ROOT / 'scripts/g3a_report.py').exists(): ROOT = ROOT.parent
spec = importlib.util.spec_from_file_location('g3a_report', ROOT / 'scripts/g3a_report.py')
report = importlib.util.module_from_spec(spec)
spec.loader.exec_module(report)
evidence = report.read_evidence()
summary = report.summarize(evidence)
display(Markdown(f"**判定：{summary['status']}**；正式运行 `{summary['run_id']}`；代码 `{summary['run_commit']}`。"))
print('数据、参数、逐题记录和源码归档哈希已核对。')

## 1. 先看运行前预测，再看结果

预测在模型运行前提交。第一次预检遇到整句填充导致的长度振荡，未训练即停止；仅对 G3a 把约 2K 明确为 2048±64 token，保留原种子、样本数和目标位置规则。后续固定配置运行，未根据留出结果挑轮数或改模型。

In [ ]:
task = (ROOT / 'docs/tasks/06_G3A_SINGLE_LAYER.md').read_text()
prediction = task.split('**运行前预测（原文保留）。**')[1].split('**正确性。**')[0]
display(Markdown('**原预测：**' + prediction))
print('预算累计秒数:', round(summary['budget']['used_seconds'], 2), '/ 7200')
print('各次运行:', [(r['run_id'], r['status']) for r in summary['budget']['runs']])

## 2. 训练到底接在哪里？

完整模型先处理资料，再逐 token 完整读取问题和模板后缀。只在第 14 层抓取四条查询的 Q 和合法历史 K/V，主体不保留反向图。其他层从未改成 D0 或 D。

每个 KV 组服务两个查询头：Q 为 `[组, 组内查询头, 查询行, 头维]`，K/V 为 `[组, 原始位置, 头维]`。索引用主 RoPE 前、归一化后的 Q/K；教师和主注意力输出使用主 RoPE 后 Q/K。两者不能互换。

In [ ]:
first = next(iter(evidence['manifest']['feature_files'].values()))
print('一份实际资料的形状:', json.dumps(first['shapes'], ensure_ascii=False, indent=2))
print('优化器仅持有:', evidence['manifest']['optimizer_parameters'])
print('可训练参数:', evidence['manifest']['trainable_parameters'])
print('主体前后哈希相同:', evidence['manifest']['body_hash_before'] == evidence['manifest']['body_hash_after_extraction'])

## 3. 教师分布不是答案标签

教师先让每个主查询头对全部合法位置做 softmax，再在同一个 KV 组内平均。候选域去掉首 4 与近 64 个固定位置。`m_C` 是候选域原本得到多少完整注意力权重；再除以它，才得到训练用的条件分布。

KL 衡量两个分布有多不同（单位 nats，越小越接近）。低于 1e-4 的候选权重行不训练，但必须保留跳过计数。完整注意力“看哪里”，不等于正确答案“必须在哪里”。

In [ ]:
source = evidence['files']['source/src/attention_lab/g3a.py'].decode()
assert hashlib.sha256((ROOT / 'src/attention_lab/g3a.py').read_bytes()).hexdigest() == evidence['manifest']['sources']['src/attention_lab/g3a.py']
def show_source(name):
    node = next(node for node in ast.parse(source).body if getattr(node, 'name', None) == name)
    display(Code(ast.get_source_segment(source, node), language='python'))
show_source('teacher')
show_source('TrainableIndexer')

索引器只更新 Wq/Wk。每次更新后，查询投影和索引 z 都重新计算；可复用的是冻结主体特征，不能复用旧参数投影出的 z。下面用同一 `.py` 的教师函数做 CPU 检查，不运行语言模型。

In [ ]:
sys.path.insert(0, str(ROOT / 'src'))
import torch
from attention_lab.g3a import teacher
q = torch.tensor([[[[1., 0.]], [[0., 1.]]]], device='cpu')
k = torch.tensor([[[1., 0.], [0., 1.], [1., 1.]]], device='cpu')
v = torch.tensor([[[2., 0.], [0., 2.], [1., 1.]]], device='cpu')
head_weights, group_mean, output = teacher(q, k, v)
print('两头各自分配:', head_weights.squeeze().tolist())
print('组平均:', group_mean.tolist())
torch.testing.assert_close(group_mean.sum(-1), torch.ones(1))
print('逐头归一化，再平均：检查通过。')

## 4. 训练损失、留出覆盖、局部误差分别看

训练固定 50 轮，只在最终参数冻结后计算留出指标。曲线是每轮更新过程中遇到的训练损失，参数在这一轮内持续变化，不能把它当作固定轮末参数的整集损失。

C 的 KL 使用均匀全候选分布，实际名单仍是均匀随机抽样；D0 是训练前，D 是最终参数，R 用昂贵的完整教师权重取 top-k。R 的 coverage 最优，不保证局部输出误差最小。

In [ ]:
display(Markdown('### 留出 16 份资料

' + report.table(summary, 'heldout')))
display(Markdown('### 训练 48 份资料：冻结参数重评

' + report.table(summary, 'train')))
import matplotlib.pyplot as plt
fig = report.plot_overview(summary)
plt.show()

点代表完整资料，不能把同一资料的 4 条查询×8 个组当作 32 个独立样本。下面保留全部留出资料的配对变化。KL 和输出误差的负值表示改善；coverage 的正值表示改善。

In [ ]:
lines=['| 资料 | ΔKL (D−D0) | Δcoverage | Δ相对输出误差 |', '|---|---:|---:|---:|']
for row in summary['paired']:
    if row['split'] == 'heldout':
        lines.append(f"| {row['doc_id']} | {row['delta_kl']:+.5f} | {row['delta_coverage']:+.5f} | {row['delta_output_error']:+.5f} |")
display(Markdown('
'.join(lines)))
print('实际读取预算范围:', summary['range'])
print('rho:', evidence['manifest']['config']['rho'])
print('k/M 范围:', min(r['k']/r['candidate_count'] for r in evidence['records']), max(r['k']/r['candidate_count'] for r in evidence['records']))
print('总读取比例范围:', min(r['selected_count']/r['legal'] for r in evidence['records']), max(r['selected_count']/r['legal'] for r in evidence['records']))

## 5. 同一道题，训练前后选中了什么？

按预登记规则，展示留出最终查询在 8 组平均 coverage 上变化最大与最差的两份资料。极值用于理解差异，不是随机抽出的典型案例。图固定展示 KV 组 0，因此图中这一组的变化不必等于 8 组平均变化。

横轴是原始 token 位置，竖线是实际选中的条目，阴影标出被问记录。首部与近期固定集合也包含在名单中。

In [ ]:
for case in summary['cases']:
    display(Markdown(f"### {case['label']}：{case['doc_id']}

{case['question']}

目标记录：`{'；'.join(case['target_records'])}`。8 组平均 coverage 变化：{case['delta_coverage']:+.5f}。"))
    lines=['| 方法（组0） | 条件coverage | 相对输出误差 | 目标记录直接选中token数 |', '|---|---:|---:|---:|']
    target={p for a,b in case['target_spans'] for p in range(a,b)}
    for row in case['records']:
        pos=set(row['positions'][0])
        lines.append(f"| {row['method']} | {row['coverage'][0]:.5f} | {row['output_relative_l2'][0]:.5f} | {len(pos & target)}/{len(target)} |")
    display(Markdown('
'.join(lines)))
    fig=report.plot_case(case)
    plt.show()

## 6. 哪条证据成立，哪条仍然没有？

训练损失下降、留出 KL 下降、选中更多教师权重、局部输出更接近完整读取，是四项不同观察。相同名额下，R 可以选到最多教师权重，但 V 的方向也会影响输出误差。

这些新资料仍使用同一生成器、提示和填充句库。留出改善可能说明学到了本分布中的稳定读取模式；不能自动解释成学会了找答案，更不能声称泛化到其他提示、长度、层或模型。

本轮没有稀疏自由生成、整模型任务分数、速度测量、全层训练或从资料阶段开始稀疏。完整代码、失败预检、参数和全部记录均保留；下一阶段由复审后另行决定。

In [ ]:
print('正式运行：', summary['run_id'])
print('运行源代码提交：', summary['run_commit'])
print('原始记录归档：', report.ARCHIVE.relative_to(ROOT))
print('初始/最终索引参数：', summary['run_id'] + '/index_initial.npz', summary['run_id'] + '/index_final.npz')
print('完整冻结特征：本机 runs/' + summary['run_id'] + '/features/；每份哈希在 manifest。')
print('精简报告重建：python scripts/g3a_report.py（无 torch 依赖）。')